# 01 — Hosted Lightning and Ultra Text2SQL targets

This CPU-only notebook evaluates NVIDIA-hosted Nemotron 3.5 Lightning and
Nemotron 3 Ultra on the same frozen BIRD Mini-Dev SQLite questions. The default
is 25 requests per model—50 total. Responses checkpoint after every success and
resume safely after a 429 or interruption.

The primary score is execution accuracy. Hosted NVFP4 services are useful task
targets, but the causal fine-tuning comparison is always a local BF16 base versus
its own merged LoRA checkpoint in Notebooks 02–04.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
print('Repository:', ROOT)
print('Artifacts:', ARTIFACTS_DIR)


## 1. API preflight and public/private endpoint switch

Copy `config/api.local.toml.example` to the ignored `config/api.local.toml` for
an internal OpenAI-compatible endpoint. Set `API_PROFILE_OVERRIDE` to `public` or
`local`; leave it `None` for automatic selection. Keys never belong in TOML.


In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'api'], check=True)
from nemotron_ft_lab.api_config import load_nvidia_api_config

API_PROFILE_OVERRIDE = None
API_CONFIG = load_nvidia_api_config(ROOT, profile=API_PROFILE_OVERRIDE)
print('API profile:', API_CONFIG.profile_name, f'({API_CONFIG.source_label})')
print('Endpoint:', API_CONFIG.base_url)
print('Lightning:', API_CONFIG.lightning.model_id, '->', API_CONFIG.lightning.served_variant)
print('Ultra:', API_CONFIG.ultra.model_id, '->', API_CONFIG.ultra.served_variant)
print('Pacing:', API_CONFIG.requests_per_minute, 'requests/minute')


## 2. Freeze the official executable Mini-Dev evaluation bundle


In [ ]:
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py',
    '--output-dir', str(DATA_DIR), '--evaluation-only',
], check=True)

from nemotron_ft_lab.constants import DEFAULT_API_EVALUATION_SIZE, DEFAULT_SEED
from nemotron_ft_lab.data import balanced_evaluation_subset, build_messages, read_jsonl

all_eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
CLOUD_EVAL_SIZE = DEFAULT_API_EVALUATION_SIZE
cloud_rows = balanced_evaluation_subset(all_eval_rows, size=CLOUD_EVAL_SIZE, seed=DEFAULT_SEED + 1)
manifest = json.loads((DATA_DIR / 'evaluation_manifest.json').read_text())
print('Frozen local holdout:', len(all_eval_rows))
print('Cloud rows per model:', len(cloud_rows))
print('Cloud requests across Lightning + Ultra:', 2 * len(cloud_rows))
print('Difficulty mix:', manifest['difficulty_distribution'])
print('Databases:', len(manifest['database_distribution']))
print(build_messages(cloud_rows[0])[1]['content'][:1800])


## 3. Authenticate without storing the key

Prefer exporting `NVIDIA_API_KEY` before Jupyter starts. Native `input()` is the
fallback because some password widgets block paste; input is briefly visible and
then cleared.


In [ ]:
from IPython.display import clear_output
from openai import OpenAI

api_key = os.environ.get('NVIDIA_API_KEY', '').strip()
used_native_prompt = not api_key
if used_native_prompt:
    api_key = input('Paste NVIDIA API key (visible until Enter), then press Enter: ').strip()
    clear_output(wait=False)
if not api_key:
    raise RuntimeError('An NVIDIA API key is required for hosted baselines.')
client = OpenAI(
    base_url=API_CONFIG.base_url, api_key=api_key,
    max_retries=0, timeout=API_CONFIG.timeout_seconds,
)
del api_key
print('API client configured; prompt cleared.' if used_native_prompt else 'API client configured from NVIDIA_API_KEY.')


## 4. Run both hosted models and execute their generated SQL


In [ ]:
from nemotron_ft_lab.evaluation import (
    generate_nvidia_api_predictions,
    paired_execution_comparison,
    save_report,
    score_predictions,
)

CLOUD_MODELS = {'lightning': API_CONFIG.lightning, 'ultra': API_CONFIG.ultra}
api_reports = {}
for name, spec in CLOUD_MODELS.items():
    endpoint_model_fingerprint = hashlib.sha256(
        f'{API_CONFIG.base_url}\0{spec.model_id}\0{manifest["evaluation_sha256"]}\0v1'.encode()
    ).hexdigest()[:12]
    artifact_name = f'{API_CONFIG.artifact_prefix}{name}_{endpoint_model_fingerprint}'
    resume_path = ARTIFACTS_DIR / f'evaluation/api_{artifact_name}_text2sql_v1_{CLOUD_EVAL_SIZE}.jsonl'
    print(f'Evaluating {name}: {spec.model_id}')
    started = time.perf_counter()
    generated = generate_nvidia_api_predictions(
        client, cloud_rows, model=spec.model_id, resume_path=resume_path,
        requests_per_minute=API_CONFIG.requests_per_minute,
    )
    report = score_predictions(generated, data_dir=DATA_DIR)
    report.update({
        'wall_time_seconds': time.perf_counter() - started,
        'endpoint': API_CONFIG.base_url, 'api_profile': API_CONFIG.profile_name,
        'served_variant': spec.served_variant, 'precision': 'NVFP4',
        'prompt_protocol': 'bird-schema-question-evidence-v1',
    })
    report_path = ARTIFACTS_DIR / f'evaluation/baseline_api_{artifact_name}_text2sql_{CLOUD_EVAL_SIZE}.json'
    save_report(report_path, report, model=spec.model_id, run_type=f'hosted-{name}-text2sql')
    api_reports[name] = report

metrics = ('n', 'execution_accuracy', 'sql_valid_rate', 'sql_executable_rate', 'normalized_exact_match')
print({name: {key: report[key] for key in metrics} for name, report in api_reports.items()})


In [ ]:
comparison = paired_execution_comparison(api_reports['lightning'], api_reports['ultra'])
comparison['interpretation'] = 'Ultra minus Lightning on identical hosted requests'
print(json.dumps(comparison, indent=2))
for name, report in api_reports.items():
    print(f'\n{name.title()} examples:')
    for row in report['rows'][:3]:
        print('\nQ:', row['question'])
        print('Gold:', row['expected_sql'])
        print('Generated:', row['generated'])
        print('Execution correct:', row['execution_correct'])


## Result contract

These scores are secondary targets. Run Notebook 02 once for each local profile
you intend to tune. Notebook 03 compares Nano with Nano; Notebook 04 compares
Lightning with Lightning. Cross-model comparisons use only shared IDs and are
labeled as context rather than causal fine-tuning evidence.
